imports

In [1]:
import clip
import json
import requests
import torch
import faiss
from tqdm import tqdm
import numpy as np
from PIL import Image
from io import BytesIO

# E(TL)
- loads COCO annotation file into memory
- defines function that fetches any image by URL into a PIL Image object
(extract step in ETL)

In [2]:
with open('data/annotations/instances_val2017.json', 'r') as file:
    # Parse the JSON file into a Python object
    data = json.load(file)

# print(data["images"][0]["coco_url"])
    
images = data["images"]

def fetch_image(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    return img

# print(fetch_image(images[0]["coco_url"]).size)
# print(fetch_image(images[0]["coco_url"]).mode)
# fetch_image(images[0]["coco_url"]).show()

- load CLIP model (ViT-B/32) 
- model encodes images into vectors
- preprocess resizes/normalizes vectors into the format CLIP expects

In [3]:

model, preprocess = clip.load("ViT-B/32", device="cpu")

In [4]:
# testing that function works as desired
# test_img = fetch_image(images[0]["coco_url"])

- single-image embedding function
- PIL Image --> returns (1, 512) numpy array
redudant and never used in ETL loop

In [5]:
# def embed_image(pil_image): 
#     img_tensor = preprocess(pil_image).unsqueeze(0)
#     # print(img_tensor.shape)

#     with torch.no_grad(): 
#         embedding = model.encode_image(img_tensor)

#     embedding_np = embedding.detach().cpu().numpy()

#     return embedding_np
#     # print(embedding.shape)
#     # print(embedding_np.shape)


# # embed_image(test_img) 

# ETL Loop
- Extract: fetch_image() pulls each image from COCO's servers
- Transform: preprocess() + torch.stack() + model.encode_image() converts batches of 32 images into (32, 512) embedding tensors
- Load: results accumulate in all_embeddings, then np.vstack() combines them into the final (512, 512) matrix

In [6]:
all_embeddings = []
metadata = []

for i in tqdm(range(0, 500, 32)):
    # images[i:i+32]
    batch_tensors = []
    for img in images[i:i+32]:
        try: 
            img_url = img["coco_url"]
            pil_image = fetch_image(img_url)                            # E
            embedding = preprocess(pil_image)
            batch_tensors.append(embedding)
            metadata.append({"id": img["id"], "coco_url": img_url})
        except Exception as e: 
            print(f"Skipped {img_url}: {e}")
            continue

    if len(batch_tensors) == 0:
        continue

    stack = torch.stack(batch_tensors)
    # print(stack.shape)

    with torch.no_grad(): 
        embedding = model.encode_image(stack)
        
    embedding_np = embedding.detach().cpu().numpy()
    # print(embedding_np.shape)  
    all_embeddings.append(embedding_np)

embeddings_matrix = np.vstack(all_embeddings)
embeddings_matrix = embeddings_matrix.astype('float32')
print(embeddings_matrix.shape)  

100%|██████████| 16/16 [02:13<00:00,  8.32s/it]

(512, 512)


In [7]:
# cleanup memeory 
import gc
del all_embeddings
del stack
del embedding
del batch_tensors
import gc
gc.collect()

0

- save artifacts embeddings_matrix.npy and metadata.json
- the ETL runs once and writes these files, the serving layer reads them without re-running the ETL

In [9]:
np.save('embeddings_matrix.npy', embeddings_matrix)
import json
with open('metadata.json', 'w') as f:
    json.dump(metadata, f)

- build FAISS index
- loads matrix, normalizes vectors, builds IndexFlatIP, runs the sanity check 
- then saves coco.index 

In [1]:
import numpy as np, faiss
embeddings_matrix = np.load('embeddings_matrix.npy')

In [2]:
faiss.normalize_L2(embeddings_matrix)
line = embeddings_matrix[0].reshape(1, 512)

d = 512

index = faiss.IndexFlatIP(d)
index.add(embeddings_matrix)   
print(index.ntotal)            

query = embeddings_matrix[0].reshape(1, 512)  # search with just one
D, I = index.search(query, 5)
print(D)  # distances — first score should be ~1.0
print(I)  # indices — expected first result is  0 (itself)        

512
[[1.0000001 0.6991873 0.6874891 0.6856581 0.6600523]]
[[  0 134 485 487 243]]


In [3]:
faiss.write_index(index, 'coco.index')